# 🚀 DeepSeek-R1 GRPO: Fine-Tuning SmolLM-135M on Google Colab
**Author:** Burla Tushar Chandramouli  
**Repository:** [github.com/tushar152006/grpo-pytorch](https://github.com/tushar152006/grpo-pytorch)  

This interactive notebook demonstrates **Group Relative Policy Optimization (GRPO)** authored strictly from mathematical first principles in native PyTorch, applied to an open-weights pretrained language model (**`HuggingFaceTB/SmolLM-135M`**) on a free Google Colab T4 GPU.

### Step 1: Environment Setup & Clone Repository

In [ ]:
# Install dependencies
!pip install -q torch transformers accelerate pytest

# Clone the clean research repository
!git clone https://github.com/tushar152006/grpo-pytorch.git
%cd grpo-pytorch

### Step 2: Verify Mathematical Invariants
Before training, run our automated verification suite to prove tensor shapes, zero-variance stability, and autograd gradient descent directions.

In [ ]:
!pytest tests/ -v

### Step 3: Load Pretrained Weights (SmolLM-135M)
We load both the **trainable Policy model** ($\pi_\theta$) and the **frozen Reference model** ($\pi_{\text{ref}}$) to compute Schulman's $k_3$ KL divergence penalty.

In [ ]:
import torch
from src.hf_adapter import load_policy_and_reference_models, format_reasoning_prompt, batch_tokenize_prompts
from src.trainer import GRPOTrainer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing on compute device: {device}")

# Load policy and reference models
policy_model, ref_model, tokenizer = load_policy_and_reference_models(
    model_name="HuggingFaceTB/SmolLM-135M",
    device=device,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)
print("Models successfully initialized!")

### Step 4: Run the Online GRPO Reasoning Training Loop
We instantiate our native `GRPOTrainer` with group size $G = 4$, surrogate clipping $\epsilon = 0.2$, and Schulman KL weight $\beta = 0.04$.

In [ ]:
optimizer = torch.optim.AdamW(policy_model.parameters(), lr=2e-5, weight_decay=0.01)

trainer = GRPOTrainer(
    policy_model=policy_model,
    ref_model=ref_model,
    optimizer=optimizer,
    group_size=4,
    max_new_tokens=32,
    clip_eps=0.2,
    kl_weight=0.04,
    temperature=0.8,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

# Sample verifiable reasoning tasks
tasks = [
    ("What is 15 + 18?", "33"),
    ("What is 8 * 7?", "56"),
    ("What is 50 - 24?", "26"),
    ("What is 9 * 6?", "54"),
]

print(f"{'Step':<6} | {'Total Loss':<12} | {'Mean Reward':<12} | {'KL Drift':<10} | {'Clip Fraction':<12}")
print("-" * 60)

for step, (question, target) in enumerate(tasks, 1):
    prompt_str = format_reasoning_prompt(question)
    prompt_ids, prompt_mask = batch_tokenize_prompts([prompt_str], tokenizer, device)
    
    metrics = trainer.train_step(
        prompt_ids=prompt_ids,
        prompt_mask=prompt_mask,
        ground_truths=[target],
        decode_fn=tokenizer.decode,
    )
    
    print(
        f"{step:<6} | "
        f"{metrics['loss/total']:<12.4f} | "
        f"{metrics['reward/mean']:<12.3f} | "
        f"{metrics['loss/kl']:<10.4f} | "
        f"{metrics['policy/clip_fraction']:<12.3f}"
    )

### Step 5: Test-Time Reasoning Generation
Evaluate the fine-tuned policy to inspect emerging `<think>` reasoning traces.

In [ ]:
test_prompt = format_reasoning_prompt("What is 12 + 19?")
test_ids, test_mask = batch_tokenize_prompts([test_prompt], tokenizer, device)

with torch.no_grad():
    outputs = policy_model.generate(
        test_ids,
        attention_mask=test_mask,
        max_new_tokens=48,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )

print("\n--- Generated Reasoning Output ---")
print(tokenizer.decode(outputs[0].tolist(), skip_special_tokens=True))
